# Занятие 3, демо 1. Хранить и читать - разные координаты

«Какая архитектура дешевле» - вопрос недоопределённый, пока не сказано, какую
координату мерить или как их складывать.

Здесь две координаты, обе на один сгенерированный токен:

- **хранение** - что лежит между шагами декодирования;
- **чтение** - сколько из этого надо прочитать, чтобы выдать следующий токен.

Размерности знакомые: 32 слоя, 32 головы запроса, 8 KV-голов, размерность
головы 128, bfloat16, одна последовательность.

**Модель идеализированная.** Считается один проход по хранимому. Записи нового
KV, обратная запись состояния, поиск нужных блоков у sparse и стоимость самих
вычислений не учтены. Это не задержка и не полный трафик памяти.

In [ ]:
import torch

torch.set_num_threads(1)

"""Вектор стоимости: сколько механизм хранит и сколько читает.

Две разные координаты. Хранение - это persistent state, то, что лежит между
шагами декодирования. Чтение - сколько байтов кэша или состояния нужно
прочитать, чтобы сгенерировать один токен.

Модель идеализированная и это важно: считается один проход по хранимому,
на одну последовательность, batch = 1. Записи нового KV, обратная запись
состояния, поиск нужных блоков у sparse и работа самих вычислений не учтены.
Это не задержка и не полный трафик памяти.
"""

GiB = 1024 ** 3
MiB = 1024 ** 2


def human(nbytes):
    """Байты в читаемом виде."""
    if nbytes >= GiB:
        return f"{nbytes / GiB:.1f} ГиБ"
    return f"{nbytes / MiB:.0f} МиБ"


def kv_cache_bytes(L, n_layers, n_kv, d_head, dtype_bytes=2):
    """KV-кэш: длина x слои x KV-головы x размерность x байты x две матрицы."""
    return L * n_layers * n_kv * d_head * dtype_bytes * 2


def state_bytes(n_layers, n_v_heads, d_k, d_v, dtype_bytes=2):
    """Конечное состояние, по одной матрице на голову значений.

    Состояние должно быть той же ширины, что и слой рядом: одна матрица
    d_k x d_v на слой была бы уже слоя внимания в число голов раз.
    """
    return n_layers * n_v_heads * d_k * d_v * dtype_bytes


# размерности одной знакомой модели: 32 слоя, 32 головы запроса,
# 8 KV-голов, размерность головы 128, bfloat16
CONFIG = dict(n_layers=32, n_q=32, n_kv=8, d_head=128, dtype_bytes=2)


def mechanisms(L, sparse_read_frac=1 / 16, cfg=CONFIG):
    """Хранение и чтение на один сгенерированный токен, в байтах."""
    full = kv_cache_bytes(L, cfg["n_layers"], cfg["n_q"], cfg["d_head"],
                          cfg["dtype_bytes"])
    gqa = kv_cache_bytes(L, cfg["n_layers"], cfg["n_kv"], cfg["d_head"],
                         cfg["dtype_bytes"])
    state = state_bytes(cfg["n_layers"], cfg["n_q"], cfg["d_head"],
                        cfg["d_head"], cfg["dtype_bytes"])
    return [
        ("full attention", full, full),
        (f"GQA, {cfg['n_kv']} KV-голов", gqa, gqa),
        (f"sparse, читает {sparse_read_frac:.4g}", full, full * sparse_read_frac),
        ("конечное состояние", state, state),
    ]


def table(L, sparse_read_frac=1 / 16, cfg=CONFIG):
    """Печатает «хранит / читает» для одной длины контекста."""
    rows = mechanisms(L, sparse_read_frac, cfg)
    width = max(len(name) for name, _, _ in rows)
    print(f"  {'механизм':<{width}}  {'хранит':>10}  {'читает':>10}")
    for name, stored, read in rows:
        print(f"  {name:<{width}}  {human(stored):>10}  {human(read):>10}")
    return rows


def read_threshold(cfg=CONFIG):
    """При какой доле чтения sparse начинает читать меньше, чем GQA.

    GQA читает долю n_kv / n_q от полного кэша. Значит sparse выигрывает по
    чтению тогда и только тогда, когда его доля меньше этой.
    """
    return cfg["n_kv"] / cfg["n_q"]

## Четыре механизма на длинном контексте

In [ ]:
for L, label in ((8192, "8K"), (131072, "128K")):
    print(f"контекст {label}")
    table(L)
    print()

## Где здесь настоящий обмен

Сравним GQA и sparse. GQA уменьшает **и** хранение, и чтение - ровно в
$n_q/n_{kv}$ раз. Sparse оставляет весь пул на месте и уменьшает только чтение.

Значит sparse читает меньше GQA тогда и только тогда, когда его доля меньше
порога

$$
f^{*}=\frac{n_{kv}}{n_q}
$$

Ниже порога механизмы несравнимы: один хранит меньше, другой читает меньше.
Выше - GQA лучше по обеим координатам, и обмена нет.

In [ ]:
print(f"порог доли чтения: f* = {read_threshold():.2f}\n")

for fraction in (1 / 2, 1 / 4, 1 / 16):
    rows = dict((name.split(",")[0], (s, r)) for name, s, r in
                mechanisms(131072, sparse_read_frac=fraction))
    gqa_store, gqa_read = rows["GQA"]
    sp_store, sp_read = rows["sparse"]
    verdict = ("несравнимы" if sp_read < gqa_read and sp_store > gqa_store
               else "GQA лучше по обеим")
    print(f"  доля {fraction:6.4f}:  sparse читает {human(sp_read):>9},"
          f"  GQA читает {human(gqa_read):>9}   -> {verdict}")

## Конечное состояние - другая история

Его размер от длины контекста **не зависит вовсе**: $O(1)$ против $O(L)$ у
любого KV-пула. На 8K разрыв со сжатым состоянием невелик, на 128K он
становится решающим - но это не обмен между координатами, а другая зависимость
от длины.

Заметьте: в показанных двух координатах конечное состояние дешевле sparse и по
хранению, и по чтению. Значит выбор между ними этими двумя числами **не
объясняется**. Разница в том, что именно хранится: состояние сжимает историю,
а KV-пул держит её пословно и допускает адресное извлечение. Этой оси в таблице
нет.

Отсюда осторожный вывод: само наличие attention-слоёв в гибриде не является
свидетельством того, что слои с конечным состоянием «не сработали». Они могут
занимать разные места и по стоимости, и по тому, что умеют доставать из
прошлого.